In [2]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import seaborn as sns

import scipy.stats as stats
from scipy.stats import spearmanr, hypergeom

from statsmodels.stats.multitest import multipletests

# GO enrichment
import gseapy as gp

## CellOracle
import celloracle as co

## Load data

In [4]:
## DATA (CellOracle object before fitting GRN)
oracle = co.load_hdf5("../data/celloracle_data/celloracle_unfit.celloracle.oracle")

# Load object
links = co.load_hdf5("../data/celloracle_data/celloracle_links_raw.celloracle.links")

Inspect loaded objects:

In [5]:
oracle

Oracle object

Meta data
    celloracle version used for instantiation: 0.20.0
    n_cells: 1604
    n_genes: 2100
    cluster_name: leiden_annotated
    dimensional_reduction_name: X_umap
    n_target_genes_in_TFdict: 1941 genes
    n_regulatory_in_TFdict: 809 genes
    n_regulatory_in_both_TFdict_and_scRNA-seq: 809 genes
    n_target_genes_both_TFdict_and_scRNA-seq: 1941 genes
    k_for_knn_imputation: 40
Status
    Gene expression matrix: Ready
    BaseGRN: Ready
    PCA calculation: Done
    Knn imputation: Done
    GRN calculation for simulation: Not finished

In [6]:
links

## FIT GRN — RIDGE REGRESSION PER CLUSTER

Use Links object to store raw GRNs per cluster (with weights and associated p-values):

In [ ]:
""""
# Infer GRN per cluster with Ridge Regression
# alpha: regularization strength. Higher = sparser network.
# verbose_level=10 prints progress per cluster
links = oracle.get_links(
    cluster_name_for_GRN_unit='leiden_annotated',
    alpha=10,
    verbose_level=10
)

# Save Links object
links.to_hdf5(file_path="../data/celloracle_data/celloracle_links_raw.celloracle.links")
"""

  0%|          | 0/6 [00:00<?, ?it/s]

Inferring GRN for ASO_KO...


  0%|          | 0/1941 [00:00<?, ?it/s]

Inferring GRN for Ectoderm_1...


  0%|          | 0/1941 [00:00<?, ?it/s]

...

  0%|          | 0/1941 [00:00<?, ?it/s]

Inferring GRN for Mesoderm_2...


  0%|          | 0/1941 [00:00<?, ?it/s]

Inferring GRN for mESCs...


  0%|          | 0/1941 [00:00<?, ?it/s]

Inspect the Links object:

In [7]:
links.links_dict.keys()

dict_keys(['ASO_KO', 'Ectoderm_1', 'Ectoderm_2', 'Mesoderm_1', 'Mesoderm_2', 'mESCs'])

In [8]:
links.links_dict["ASO_KO"]

,source,target,coef_mean,coef_abs,p,-logp
0,E2f2,1110051M20Rik,-0.004956,0.004956,1.581332e-01,0.800977
1,Klf16,1110051M20Rik,0.085048,0.085048,6.993005e-07,6.155336
2,Nfe2,1110051M20Rik,-0.010746,0.010746,3.024051e-03,2.519411
3,Klf1,1110051M20Rik,0.002166,0.002166,1.519522e-01,0.818293
4,Crebzf,1110051M20Rik,-0.014760,0.014760,1.886380e-03,2.724371
...,...,...,...,...,...,...
719588,Rarb,Zzef1,0.021106,0.021106,4.898473e-10,9.309939
719589,Tfap2a,Zzef1,0.011528,0.011528,1.918989e-07,6.716927
719590,Vdr,Zzef1,-0.005756,0.005756,9.833774e-04,3.007280
719591,Elf2,Zzef1,-0.003766,0.003766,3.479081e-02,1.458535


## Plots to decide filtering


In [ ]:
## Auxiliar plot functions:

def plot_dual_boxplot(ax, data, x_col, title, x_label):

    # Color palette: blue (inward), red (outward)
    my_pal = {'Inward': '#1f77b4', 'Outward': '#d62728'}

    # Boxplot with dodge=True (side by side)
    sns.boxplot(
        data=data, x=x_col, y='Edges', hue='Direction',
        palette=my_pal, showfliers=False, dodge=True, ax=ax, boxprops=dict(alpha=0.5)
    )
    # Stripplot for the points with dodge=True to align with boxes
    sns.stripplot(
        data=data, x=x_col, y='Edges', hue='Direction',
        palette=my_pal, dodge=True, jitter=0.2, alpha=0.7, size=5, ax=ax, edgecolor='black', linewidth=0.5
    )
    
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel(x_label, fontsize=12)
    ax.set_ylabel("Number of Edges", fontsize=12)
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    ax.set_yscale('log')
    ax.set_ylim(bottom=1)

    # Fix legend (Seaborn duplicates legend for boxplot and stripplot)
    handles, labels = ax.get_legend_handles_labels()
    # Take only the first two handles (Inward, Outward)
    ax.legend(handles[:2], labels[:2], title="Edge Direction")


# ----------------------------------------------------------------------------------------

def plot_cluster_dynamics(ax, data, title, x_label):
    # Background Boxplot: Shows the macrostate variance across all clusters
    sns.boxplot(
        data=data, x='Threshold', y='Edges',
        color='lightgray', showfliers=False, ax=ax, boxprops=dict(alpha=0.5)
    )
    
    # Stripplot: Colored by specific Cluster (Tracking microstates)
    sns.stripplot(
        data=data, x='Threshold', y='Edges', hue='Cluster',
        palette='tab10', dodge=False, jitter=0.15, alpha=0.9, size=7, 
        ax=ax, edgecolor='black', linewidth=0.5
    )
    
    # Aesthetics
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel(x_label, fontsize=11)
    ax.set_ylabel("Number of Edges", fontsize=11)
    ax.grid(axis='y', linestyle='--', alpha=0.5)

    ax.set_yscale('log')
    ax.set_ylim(bottom=1)
    
    # Move legend outside the plot
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', title="Cluster", fontsize=9)

General

In [6]:
# We iterate over the UNFILTERED raw results (links_dict) to see everything
clusters = list(links.links_dict.keys())

# Setup grid for the subplots based on number of clusters
n_clusters = len(clusters)
n_cols = 3
n_rows = int(np.ceil(n_clusters / n_cols))

# List of p-values to test (from mild to extremely strict)
p_thresholds = [1e-2, 1e-4, 1e-6, 1e-7, 1e-8, 1e-9,1e-10]

# Define a range of beta absolute values to test (from very weak to strong links)
coef_thresholds = [0.001, 0.005, 0.010, 0.015, 0.025, 0.040, 0.050]

Check how the regression affect the number of edges and coefficient values for different p-values:

In [ ]:
# =====================================================================
# EVALUATION OF NETWORK SIZE VS P-VALUE THRESHOLD
# =====================================================================

edge_counts = []

for p_val in p_thresholds:
    # Filter without a fixed number cap
    links.filter_links(p=p_val, threshold_number=None)
    
    # Record the number of edges for each cluster at this p-value
    for cluster, df_filtered in links.filtered_links.items():
        count = len(df_filtered) if df_filtered is not None else 0
        edge_counts.append({
            'P_value': p_val,
            '-Log10(P)': -np.log10(p_val),
            'Cluster': cluster,
            'Edges': count
        })

df_edges = pd.DataFrame(edge_counts)

# --- PLOT 1: BOXPLOT OF EDGE COUNTS VS P-VALUE ---
fig1, ax1 = plt.subplots(figsize=(10, 6))

# Boxplot to show the median and variance across clusters for each p-value
sns.boxplot(
    data=df_edges, 
    x='-Log10(P)', 
    y='Edges', 
    color='lightblue', 
    showfliers=False, 
    ax=ax1
)
# Add swarmplot to see individual clusters as dots on top of the boxplots
sns.swarmplot(
    data=df_edges, 
    x='-Log10(P)', 
    y='Edges', 
    hue='Cluster', 
    palette='Set2', 
    size=6, 
    ax=ax1
)

ax1.set_title("Evolution of GRN total edges vs p-val threshold", fontsize=14, fontweight='bold')
ax1.set_xlabel("-log10 threshold", fontsize=12, fontweight='bold')
ax1.set_ylabel("GRN total edges", fontsize=12, fontweight='bold')
ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax1.grid(True, axis='y', linestyle='--', alpha=0.5)

ax1.set_yscale('log')
ax1.set_ylim(bottom=1)

plt.tight_layout()
fig1.savefig("../Plots/GRN_construction/GRN_edges_vs_pval.png", bbox_inches='tight')

# =====================================================================
# SCATTER PLOTS: COEF ABS VS P-VALUE (VOLCANO STYLE)
# =====================================================================

fig2, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows), sharex=True, sharey=True)
if n_clusters > 1:
    axes = axes.flatten()
else:
    axes = [axes]

for i, cluster in enumerate(clusters):
    ax = axes[i]
    df_raw = links.links_dict[cluster]
    
    if df_raw is None or len(df_raw) == 0:
        ax.set_title(f"{cluster} (Empty)")
        continue
    
    # Extract variables
    # We clip p-values to 1e-300 to avoid log10(0) = -infinity errors
    p_vals = df_raw['p'].clip(lower=1e-300)
    neg_log_p = -np.log10(p_vals)
    coef_abs = df_raw['coef_abs']
    coef_mean = df_raw['coef_mean'] # Original coefficient to check sign
    
    # Assign colors: Red if negative correlation, Green if positive
    colors = np.where(coef_mean < 0, 'red', 'green')
    
    # Plot Scatter
    ax.scatter(neg_log_p, coef_abs, c=colors, alpha=0.1, s=3)
    
    # Aesthetics
    ax.set_title(f"Cluster:{cluster}", fontsize=12, fontweight='bold')
    ax.set_xlabel("-log10(p-val)")
    ax.set_ylabel("$|\\beta|$ threshold")
    ax.grid(True, linestyle='--', alpha=0.3)
    
    # Draw reference line for p=1e-8 (-log10 = 8)
    ax.axvline(8, color='black', linestyle='--', alpha=0.5, label='p=1e-8')
    ax.legend(loc='upper left', fontsize=8)

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
fig2.savefig("../Plots/GRN_construction/GRN_coef_vs_pval.png", bbox_inches='tight', dpi=300)

2026-07-13 15:10:29,413 - INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-07-13 15:10:29,419 - INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-07-13 15:10:29,460 - INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-07-13 15:10:29,466 - INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


We will check negative and positive edge distribution with p_value to confirm that POSITIVE EDGES HAVE HIGHER Pval

In [16]:
# =====================================================================
# OVERLAPPED HISTOGRAMS: POSITIVE VS NEGATIVE EDGES
# =====================================================================
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows), sharex=True, sharey=True)
fig.suptitle("Distribution of Edges vs -log10(p-value) (Positive vs Negative)", fontsize=18, fontweight='bold')

if n_clusters > 1:
    axes = axes.flatten()
else:
    axes = [axes]

for i, cluster in enumerate(clusters):
    ax = axes[i]
    df_raw = links.links_dict[cluster]
    
    if df_raw is None or len(df_raw) == 0:
        ax.set_title(f"Cluster: {cluster} (Empty)", fontsize=12, fontweight='bold')
        continue
    
    # Isolate Negative and Positive Edges
    df_neg = df_raw[df_raw['coef_mean'] < 0].copy()
    df_pos = df_raw[df_raw['coef_mean'] > 0].copy()
    
    # Plot Negative Edges (Red)
    if len(df_neg) > 0:
        p_vals_neg = df_neg['p'].clip(lower=1e-300)
        neg_log_p_neg = -np.log10(p_vals_neg)
        sns.histplot(
            x=neg_log_p_neg, bins=50, color='red', ax=ax, 
            kde=True, element='step', alpha=0.3, label='Negative ($\\beta < 0$)'
        )
        
    # Plot Positive Edges (Green)
    if len(df_pos) > 0:
        p_vals_pos = df_pos['p'].clip(lower=1e-300)
        neg_log_p_pos = -np.log10(p_vals_pos)
        sns.histplot(
            x=neg_log_p_pos, bins=50, color='green', ax=ax, 
            kde=True, element='step', alpha=0.3, label='Positive ($\\beta > 0$)'
        )
    
    # Aesthetics
    ax.set_title(f"Cluster: {cluster}", fontsize=12, fontweight='bold')
    ax.set_xlabel("-log10(p-val)")
    ax.set_ylabel("Edge Count")
    ax.grid(True, linestyle='--', alpha=0.3)
    
    # Draw reference line for p=1e-8
    ax.axvline(8, color='black', linestyle='--', alpha=0.5, label='p=1e-8')
    
    # Legend
    ax.legend(loc='upper right', fontsize=8)

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
fig.savefig("../Plots/GRN_construction/GRN_edge_sign_vs_pval_hist.png", bbox_inches='tight', dpi=300)

Filter by absolute threshold and evaluate the effect on the number of edges:

In [ ]:
# =====================================================
# EVALUATION OF NETWORK SIZE VS COEFFICIENT THRESHOLD
# =====================================================

edge_counts_coef = []

# We keep a basic statistical safeguard (p < 0.01) to remove pure noise,
# but the main driving filter will be your physical coefficient threshold.
BASE_P_VAL = 0.01

for coef_th in coef_thresholds:
    for cluster, df_raw in links.links_dict.items():
        if df_raw is None or len(df_raw) == 0:
            count = 0
        else:
            # Apply your physical logic: |beta| >= threshold AND p < 0.01
            mask = (df_raw['coef_abs'] >= coef_th) & (df_raw['p'] < BASE_P_VAL)
            count = mask.sum()
            
        edge_counts_coef.append({
            'Coef_Threshold': coef_th,
            'Cluster': cluster,
            'Edges': count
        })

df_edges_coef = pd.DataFrame(edge_counts_coef)

# =====================================================================
# 2. PLOTTING THE THRESHOLD DYNAMICS (BOXPLOT)
# =====================================================================

fig, ax = plt.subplots(figsize=(10, 6))

# Boxplot shows the median network size drop across all clusters
sns.boxplot(
    data=df_edges_coef, 
    x='Coef_Threshold', 
    y='Edges', 
    color='lightgreen', 
    showfliers=False, 
    ax=ax
)

# Swarmplot overlays the individual clusters to see the physical asymmetry
sns.swarmplot(
    data=df_edges_coef, 
    x='Coef_Threshold', 
    y='Edges', 
    hue='Cluster', 
    palette='Set2', 
    size=6, 
    ax=ax
)

ax.set_title("Evolution of network total # of edges with $|\\beta|$ threshold", fontsize=14, fontweight='bold')
ax.set_xlabel("$|\\beta|$ threshold", fontsize=12, fontweight='bold')
ax.set_ylabel("GRN total edges", fontsize=12, fontweight='bold')

# Move legend outside
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, axis='y', linestyle='--', alpha=0.5)

ax.set_yscale('log')
ax.set_ylim(bottom=1)

plt.tight_layout()
fig.savefig("../Plots/GRN_construction/GRN_edges_vs_coeff.png", bbox_inches='tight')

2026-07-13 15:11:37,147 - INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-07-13 15:11:37,152 - INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-07-13 15:11:37,191 - INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-07-13 15:11:37,196 - INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


Check the evolution of inward and outward edges to the lncRNAs with the pvalue and abs_coefficient:

In [7]:
# =====================================================================
# PARAMETER SWEEP FOR LncRNAs (P-Value & Coefficient)
# =====================================================================

lncrnas = {
    'C13': 'C130026L21Rik',
    'RMST1': 'Rmst'
}

p_thresholds = [1e-2, 1e-4, 1e-6, 1e-8, 1e-10, 1e-12] 
coef_thresholds = [0.001, 0.005, 0.010, 0.015, 0.025, 0.040, 0.050]
BASE_P_VAL = 0.01

records_p = []
records_coef = []

# Iterate over each cluster and its corresponding raw links DataFrame
for cluster, df_raw in links.links_dict.items():
    if df_raw is None or len(df_raw) == 0:
        continue
        
    # --- A. p-val mapping---
    for p_val in p_thresholds:
        df_filt = df_raw[df_raw['p'] < p_val]
        
        for name, gen_id in lncrnas.items():
            for sign_name, sign_mask in [('Positive', df_filt['coef_mean'] > 0), ('Negative', df_filt['coef_mean'] < 0)]:
                df_sign = df_filt[sign_mask]
                inward = (df_sign['target'] == gen_id).sum()
                outward = (df_sign['source'] == gen_id).sum()
                
                records_p.append({'Threshold': -np.log10(p_val), 'LncRNA': name, 'Direction': 'Inward', 'Sign': sign_name, 'Edges': inward, 'Cluster': cluster})
                records_p.append({'Threshold': -np.log10(p_val), 'LncRNA': name, 'Direction': 'Outward', 'Sign': sign_name, 'Edges': outward, 'Cluster': cluster})

    # --- B. coeff. mapping ---
    for coef_th in coef_thresholds:
        df_filt = df_raw[(df_raw['coef_abs'] >= coef_th) & (df_raw['p'] < BASE_P_VAL)]
        
        for name, gen_id in lncrnas.items():
            for sign_name, sign_mask in [('Positive', df_filt['coef_mean'] > 0), ('Negative', df_filt['coef_mean'] < 0)]:
                df_sign = df_filt[sign_mask]
                inward = (df_sign['target'] == gen_id).sum()
                outward = (df_sign['source'] == gen_id).sum()
                
                records_coef.append({'Threshold': coef_th, 'LncRNA': name, 'Direction': 'Inward', 'Sign': sign_name,   'Edges': inward, 'Cluster': cluster})
                records_coef.append({'Threshold': coef_th, 'LncRNA': name, 'Direction': 'Outward', 'Sign': sign_name, 'Edges': outward, 'Cluster': cluster})

df_p = pd.DataFrame(records_p)
df_coef = pd.DataFrame(records_coef)

## For total edges per threshold, we can group by Threshold, LncRNA, Direction, and Cluster, summing the Edges.
df_p_total = df_p.groupby(
    ['Threshold', 'LncRNA', 'Direction', 'Cluster'], as_index=False
)['Edges'].sum()

df_coef_total = df_coef.groupby(
    ['Threshold', 'LncRNA', 'Direction', 'Cluster'], as_index=False
)['Edges'].sum()

# ===============================
# PLOTTING THE 4 DUAL-BOXPLOTS
# ===============================

# Color palette: blue (inward), red (outward)
my_pal = {'Inward': '#1f77b4', 'Outward': '#d62728'}

fig, axes = plt.subplots(2, 2, figsize=(16, 12), constrained_layout=True, sharey=True)

# 1. P-value Sweep for C13
plot_dual_boxplot(
    axes[0, 0], df_p_total[df_p_total['LncRNA'] == 'C13'], 
    'Threshold', "C13", "-Log10(P-value)"
)

# 2. P-value Sweep for RMST1
plot_dual_boxplot(
    axes[0, 1], df_p_total[df_p_total['LncRNA'] == 'RMST1'], 
    'Threshold', "RMST1", "-log10(p-value)"
)

# 3. Coef Sweep for C13
plot_dual_boxplot(
    axes[1, 0], df_coef_total[df_coef_total['LncRNA'] == 'C13'], 
    'Threshold', "C13", "$|\\beta|$ threshold"
)

# 4. Coef Sweep for RMST1
plot_dual_boxplot(
    axes[1, 1], df_coef_total[df_coef_total['LncRNA'] == 'RMST1'], 
    'Threshold', "RMST1", "$|\\beta|$ threshold"
)

fig.savefig("../Plots/GRN_construction/lncRNA_regulation_vs_thresholds.png", bbox_inches='tight')

2026-07-13 16:42:34,291 - INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-07-13 16:42:34,296 - INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-07-13 16:42:34,352 - INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-07-13 16:42:34,357 - INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-07-13 16:42:34,408 - INFO - Using categorical units to plot a list of strings that are all parsable as 

Same as before but looking at cluster distribution:

In [8]:
# ===============
# P-VALUE SWEEP 
# ===============

fig1, axes1 = plt.subplots(2, 2, figsize=(16, 10), constrained_layout=True, sharex=True, sharey=True)
fig1.suptitle("lncRNA network edges vs p-value threshold (by cluster)", fontsize=18, fontweight='bold')

# Row 0: INWARD EDGES (Upstream Regulators hitting the LncRNA)
plot_cluster_dynamics(
    axes1[0, 0], df_p_total[(df_p_total['LncRNA'] == 'C13') & (df_p_total['Direction'] == 'Inward')],
    "C13 INWARD", "-log10(p-value)"
)
plot_cluster_dynamics(
    axes1[0, 1], df_p_total[(df_p_total['LncRNA'] == 'RMST1') & (df_p_total['Direction'] == 'Inward')],
    "RMST1 INWARD", "-log10(p-value)"
)

# Row 1: OUTWARD EDGES (Downstream Targets pushed by the LncRNA)
plot_cluster_dynamics(
    axes1[1, 0], df_p_total[(df_p_total['LncRNA'] == 'C13') & (df_p_total['Direction'] == 'Outward')],
    "C13 OUTWARD", "-log10(p-value)"
)
plot_cluster_dynamics(
    axes1[1, 1], df_p_total[(df_p_total['LncRNA'] == 'RMST1') & (df_p_total['Direction'] == 'Outward')],
    "RMST1 OUTWARD", "-log10(p-value)"
)

fig1.savefig("../Plots/GRN_construction/lncRNA_edges_vs_pval_by_cluster.png", bbox_inches='tight')

# ============================
# ABSOLUTE COEFFICIENT SWEEP 
# ============================

fig2, axes2 = plt.subplots(2, 2, figsize=(16, 10), constrained_layout=True, sharex=True, sharey=True)
fig2.suptitle("lncRNA network edges vs $|\\beta|$ threshold (by cluster)", fontsize=18, fontweight='bold')

# Row 0: INWARD EDGES
plot_cluster_dynamics(
    axes2[0, 0], df_coef_total[(df_coef_total['LncRNA'] == 'C13') & (df_coef_total['Direction'] == 'Inward')],
    "C13 INWARD", "$|\\beta|$ threshold"
)
plot_cluster_dynamics(
    axes2[0, 1], df_coef_total[(df_coef_total['LncRNA'] == 'RMST1') & (df_coef_total['Direction'] == 'Inward')],
    "RMST1 INWARD", "$|\\beta|$ threshold"
)

# Row 1: OUTWARD EDGES
plot_cluster_dynamics(
    axes2[1, 0], df_coef_total[(df_coef_total['LncRNA'] == 'C13') & (df_coef_total['Direction'] == 'Outward')],
    "C13 OUTWARD", "$|\\beta|$ threshold"
)
plot_cluster_dynamics(
    axes2[1, 1], df_coef_total[(df_coef_total['LncRNA'] == 'RMST1') & (df_coef_total['Direction'] == 'Outward')],
    "RMST1 OUTWARD", "$|\\beta|$ threshold"
)

fig2.savefig("../Plots/GRN_construction/lncRNA_edges_vs_coef_by_cluster.png", bbox_inches='tight')


TypeError: plot_cluster_dynamics() missing 1 required positional argument: 'my_pal'

We are going to look specifically to the sign of the regulatory edges in the lncRNAs, i.e. same figure as before but with two figures per case:

In [13]:

# =======================================================
# P-VALUE SWEEP (POSITIVE EDGES: beta > 0)
# =======================================================

fig1_pos, axes1_pos = plt.subplots(2, 2, figsize=(16, 10), constrained_layout=True, sharex=True, sharey=True)
fig1_pos.suptitle("lncRNA network POSITIVE edges ($\\beta > 0$) vs p-value threshold (by cluster)", fontsize=18, fontweight='bold')

# Row 0: INWARD EDGES 
plot_cluster_dynamics(
    axes1_pos[0, 0], df_p[(df_p['LncRNA'] == 'C13') & (df_p['Direction'] == 'Inward') & (df_p['Sign'] == 'Positive')],
    "C13 INWARD (Positive)", "-log10(p-value)"
)
plot_cluster_dynamics(
    axes1_pos[0, 1], df_p[(df_p['LncRNA'] == 'RMST1') & (df_p['Direction'] == 'Inward') & (df_p['Sign'] == 'Positive')],
    "RMST1 INWARD (Positive)", "-log10(p-value)"
)

# Row 1: OUTWARD EDGES 
plot_cluster_dynamics(
    axes1_pos[1, 0], df_p[(df_p['LncRNA'] == 'C13') & (df_p['Direction'] == 'Outward') & (df_p['Sign'] == 'Positive')],
    "C13 OUTWARD (Positive)", "-log10(p-value)"
)
plot_cluster_dynamics(
    axes1_pos[1, 1], df_p[(df_p['LncRNA'] == 'RMST1') & (df_p['Direction'] == 'Outward') & (df_p['Sign'] == 'Positive')],
    "RMST1 OUTWARD (Positive)", "-log10(p-value)"
)

fig1_pos.savefig("../Plots/GRN_construction/lncRNA_POSITIVE_edges_vs_pval_by_cluster.png", bbox_inches='tight')


# =======================================================
# P-VALUE SWEEP (NEGATIVE EDGES: beta < 0)
# =======================================================

fig1_neg, axes1_neg = plt.subplots(2, 2, figsize=(16, 10), constrained_layout=True, sharex=True, sharey=True)
fig1_neg.suptitle("lncRNA network NEGATIVE edges ($\\beta < 0$) vs p-value threshold (by cluster)", fontsize=18, fontweight='bold')

# Row 0: INWARD EDGES 
plot_cluster_dynamics(
    axes1_neg[0, 0], df_p[(df_p['LncRNA'] == 'C13') & (df_p['Direction'] == 'Inward') & (df_p['Sign'] == 'Negative')],
    "C13 INWARD (Negative)", "-log10(p-value)"
)
plot_cluster_dynamics(
    axes1_neg[0, 1], df_p[(df_p['LncRNA'] == 'RMST1') & (df_p['Direction'] == 'Inward') & (df_p['Sign'] == 'Negative')],
    "RMST1 INWARD (Negative)", "-log10(p-value)"
)

# Row 1: OUTWARD EDGES 
plot_cluster_dynamics(
    axes1_neg[1, 0], df_p[(df_p['LncRNA'] == 'C13') & (df_p['Direction'] == 'Outward') & (df_p['Sign'] == 'Negative')],
    "C13 OUTWARD (Negative)", "-log10(p-value)"
)
plot_cluster_dynamics(
    axes1_neg[1, 1], df_p[(df_p['LncRNA'] == 'RMST1') & (df_p['Direction'] == 'Outward') & (df_p['Sign'] == 'Negative')],
    "RMST1 OUTWARD (Negative)", "-log10(p-value)"
)

fig1_neg.savefig("../Plots/GRN_construction/lncRNA_NEGATIVE_edges_vs_pval_by_cluster.png", bbox_inches='tight')


# =======================================================
# ABSOLUTE COEFFICIENT SWEEP (POSITIVE EDGES: beta > 0)
# =======================================================

fig2_pos, axes2_pos = plt.subplots(2, 2, figsize=(16, 10), constrained_layout=True, sharex=True, sharey=True)
fig2_pos.suptitle("lncRNA network POSITIVE edges ($\\beta > 0$) vs $|\\beta|$ threshold (by cluster)", fontsize=18, fontweight='bold')

# Row 0: INWARD EDGES
plot_cluster_dynamics(
    axes2_pos[0, 0], df_coef[(df_coef['LncRNA'] == 'C13') & (df_coef['Direction'] == 'Inward') & (df_coef['Sign'] == 'Positive')],
    "C13 INWARD (Positive)", "$|\\beta|$ threshold"
)
plot_cluster_dynamics(
    axes2_pos[0, 1], df_coef[(df_coef['LncRNA'] == 'RMST1') & (df_coef['Direction'] == 'Inward') & (df_coef['Sign'] == 'Positive')],
    "RMST1 INWARD (Positive)", "$|\\beta|$ threshold"
)

# Row 1: OUTWARD EDGES
plot_cluster_dynamics(
    axes2_pos[1, 0], df_coef[(df_coef['LncRNA'] == 'C13') & (df_coef['Direction'] == 'Outward') & (df_coef['Sign'] == 'Positive')],
    "C13 OUTWARD (Positive)", "$|\\beta|$ threshold"
)
plot_cluster_dynamics(
    axes2_pos[1, 1], df_coef[(df_coef['LncRNA'] == 'RMST1') & (df_coef['Direction'] == 'Outward') & (df_coef['Sign'] == 'Positive')],
    "RMST1 OUTWARD (Positive)", "$|\\beta|$ threshold"
)

fig2_pos.savefig("../Plots/GRN_construction/lncRNA_POSITIVE_edges_vs_coef_by_cluster.png", bbox_inches='tight')


# =======================================================
# ABSOLUTE COEFFICIENT SWEEP (NEGATIVE EDGES: beta < 0)
# =======================================================

fig2_neg, axes2_neg = plt.subplots(2, 2, figsize=(16, 10), constrained_layout=True, sharex=True, sharey=True)
fig2_neg.suptitle("lncRNA network NEGATIVE edges ($\\beta < 0$) vs $|\\beta|$ threshold (by cluster)", fontsize=18, fontweight='bold')

# Row 0: INWARD EDGES
plot_cluster_dynamics(
    axes2_neg[0, 0], df_coef[(df_coef['LncRNA'] == 'C13') & (df_coef['Direction'] == 'Inward') & (df_coef['Sign'] == 'Negative')],
    "C13 INWARD (Negative)", "$|\\beta|$ threshold"
)
plot_cluster_dynamics(
    axes2_neg[0, 1], df_coef[(df_coef['LncRNA'] == 'RMST1') & (df_coef['Direction'] == 'Inward') & (df_coef['Sign'] == 'Negative')],
    "RMST1 INWARD (Negative)", "$|\\beta|$ threshold"
)

# Row 1: OUTWARD EDGES
plot_cluster_dynamics(
    axes2_neg[1, 0], df_coef[(df_coef['LncRNA'] == 'C13') & (df_coef['Direction'] == 'Outward') & (df_coef['Sign'] == 'Negative')],
    "C13 OUTWARD (Negative)", "$|\\beta|$ threshold"
)
plot_cluster_dynamics(
    axes2_neg[1, 1], df_coef[(df_coef['LncRNA'] == 'RMST1') & (df_coef['Direction'] == 'Outward') & (df_coef['Sign'] == 'Negative')],
    "RMST1 OUTWARD (Negative)", "$|\\beta|$ threshold"
)

fig2_neg.savefig("../Plots/GRN_construction/lncRNA_NEGATIVE_edges_vs_coef_by_cluster.png", bbox_inches='tight')

2026-07-13 15:11:44,892 - INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-07-13 15:11:44,898 - INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-07-13 15:11:44,947 - INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-07-13 15:11:44,954 - INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-07-13 15:11:45,003 - INFO - Using categorical units to plot a list of strings that are all parsable as 

## Filter the links object

==> by p_value

==> by absolute coefficient threshold

In [20]:
# ======================================
# Absolute coefficient manual filtering 
# ======================================

# Replace this with the threshold you choose after looking at the plot!
# For example, 0.1 is often a very robust cutoff for Ridge Regression betas.
CHOSEN_COEF_TH = 0.10  
CHOSEN_P_TH = 0.01     # Base statistical sanity check

# We manually rebuild the filtered dictionary
links.filtered_links = {}

for cluster, df_raw in links.links_dict.items():
    if df_raw is not None and len(df_raw) > 0:
        # 1. Create the mask
        mask = (df_raw['coef_abs'] >= CHOSEN_COEF_TH) & (df_raw['p'] < CHOSEN_P_TH)
        
        # 2. Extract the clean DataFrame
        df_clean = df_raw[mask].copy()
        
        # 3. Overwrite CellOracle's internal state
        links.filtered_links[cluster] = df_clean
        
        print(f"Cluster {cluster}: Retained {len(df_clean)} edges (|beta| >= {CHOSEN_COEF_TH})")
    else:
        links.filtered_links[cluster] = None

Cluster ASO_KO: Retained 2272 edges (|beta| >= 0.1)
Cluster Ectoderm_1: Retained 2122 edges (|beta| >= 0.1)
Cluster Ectoderm_2: Retained 4001 edges (|beta| >= 0.1)
Cluster Mesoderm_1: Retained 2948 edges (|beta| >= 0.1)
Cluster Mesoderm_2: Retained 796 edges (|beta| >= 0.1)
Cluster mESCs: Retained 2436 edges (|beta| >= 0.1)


Filter the inferred GRNs by p-value (optional to also put a maximum of edges per cluster):

In [6]:
# p: maximum adjusted p-value for the Ridge coefficient
# weight: rank edges by absolute coefficient value
# threshold_number: keep top N edges per cluster (not total) => by default 1e4
# Put threshold_number=None for no limit on the number of edges per cluster
links.filter_links(
    p=1e-12,
    weight='coef_abs',
    threshold_number=None
)

# Check how many edges survive per cluster
links.filtered_links  # dict {cluster_id: DataFrame with filtered edges}
for cluster, df in links.filtered_links.items():
    print(f"Cluster {cluster}: {len(df)} edges after filtering")

# Save and load filtered Links object
#links.to_hdf5(file_path="links.celloracle.links")
#links = co.load_hdf5(file_path="links.celloracle.links")

Cluster ASO_KO: 2965 edges after filtering
Cluster Ectoderm_1: 5893 edges after filtering
Cluster Ectoderm_2: 6854 edges after filtering
Cluster Mesoderm_1: 4286 edges after filtering
Cluster Mesoderm_2: 7623 edges after filtering
Cluster mESCs: 6183 edges after filtering


Basic topology visualization: 

1. undirected degree distribution per cluster

2. centrality scores

In [39]:
# degree distribution plots for the filtered links
links.plot_degree_distributions(
    plot_model=True,
    save=None
)

ASO_KO
Ectoderm_1
Ectoderm_2
Mesoderm_1
Mesoderm_2
mESCs


In [40]:
# Calculate network scores.
links.get_network_score()
links.merged_score.head()

,degree_all,degree_centrality_all,degree_in,degree_centrality_in,degree_out,degree_centrality_out,betweenness_centrality,eigenvector_centrality,cluster
Nhlh1,20,0.012829,2,0.001283,18,0.011546,31.0,0.993445,ASO_KO
Sox11,12,0.007697,3,0.001924,9,0.005773,1231.0,1.000000,ASO_KO
Pou2f2,20,0.012829,0,0.000000,20,0.012829,0.0,0.082707,ASO_KO
Basp1,1,0.000641,1,0.000641,0,0.000000,0.0,0.033702,ASO_KO
Smad1,6,0.003849,3,0.001924,3,0.001924,1760.0,0.000006,ASO_KO


Use filtered GRN for simulation:

In [ ]:
oracle.get_cluster_specific_TFdict_from_Links(links_object=links)

oracle.fit_GRN_for_simulation(
    alpha=10,
    use_cluster_specific_TFdict=True  # uses the filtered per-cluster networks
)

## Simulate shift

Also extracts expected expression shift after KO

In [ ]:
# =============================================================
# STEP 6: SIMULATE RMST1 KNOCKOUT
# Sets Rmst expression to 0 in all cells and propagates
# the perturbation through the fitted GRN.
# =============================================================

oracle.simulate_shift(
    perturb_condition={"Rmst": 0.0},  # KO: set Rmst to zero
    n_propagation=3   # number of propagation steps through the network
)


# =============================================================
# STEP 7: COMPUTE TRANSITION PROBABILITIES AND EMBEDDING SHIFT
# Translates the gene expression shift into a probability of
# transitioning to neighboring cells in the UMAP embedding.
# =============================================================

oracle.estimate_transition_prob(
    n_neighbors=40,
    knn_random=True,
    sampled_fraction=0.5
)

oracle.calculate_embedding_shift(sigma_corr=0.05)


# =============================================================
# STEP 8: VISUALIZE VECTOR FIELD ON UMAP
# =============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Grid-based vector field (cleaner visualization)
oracle.plot_simulation_flow_on_grid(
    scale=0.4,
    ax=axes[0],
)
axes[0].set_title('RMST1 KO — vector field (grid)')

# Single-cell arrows
oracle.plot_simulation_flow_random_sampling(
    scale=0.4,
    ax=axes[1],
    color_by='leiden',
    n_each_cluster=30
)
axes[1].set_title('RMST1 KO — vector field (single cells)')

plt.tight_layout()
#plt.savefig('../figures/rmst1_ko_vector_field.pdf', dpi=150)
plt.show()


# =============================================================
# SAVE ORACLE OBJECT FOR FURTHER ANALYSIS
# =============================================================

#oracle.to_hdf5("../data/oracle_rmst1_ko.celloracle.hdf5")